Датасет: Life Expectancy Data. (Kaggle)
Данные содержат пропуски и столбцы с категориальными переменными. Также перед заполнением пропусков я масштабирую данные.
Линейная регрессия предсказывает продолжительность жизни человека.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction import FeatureHasher
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error



In [2]:
data_path = "C:/Users/HONOR/Desktop/projects/hw04_Linear_Regression/Life Expectancy Data.csv"
data = pd.read_csv(data_path)
print(data.head())
print(data.describe())
print(data.shape)
print(data.iloc[0])

       Country  Year      Status  Life expectancy   Adult Mortality  \
0  Afghanistan  2015  Developing              65.0            263.0   
1  Afghanistan  2014  Developing              59.9            271.0   
2  Afghanistan  2013  Developing              59.9            268.0   
3  Afghanistan  2012  Developing              59.5            272.0   
4  Afghanistan  2011  Developing              59.2            275.0   

   infant deaths  Alcohol  percentage expenditure  Hepatitis B  Measles   ...  \
0             62     0.01               71.279624         65.0      1154  ...   
1             64     0.01               73.523582         62.0       492  ...   
2             66     0.01               73.219243         64.0       430  ...   
3             69     0.01               78.184215         67.0      2787  ...   
4             71     0.01                7.097109         68.0      3013  ...   

   Polio  Total expenditure  Diphtheria    HIV/AIDS         GDP  Population  \
0    6.

In [3]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 2938 entries, 0 to 2937
Data columns (total 22 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Country                          2938 non-null   str    
 1   Year                             2938 non-null   int64  
 2   Status                           2938 non-null   str    
 3   Life expectancy                  2928 non-null   float64
 4   Adult Mortality                  2928 non-null   float64
 5   infant deaths                    2938 non-null   int64  
 6   Alcohol                          2744 non-null   float64
 7   percentage expenditure           2938 non-null   float64
 8   Hepatitis B                      2385 non-null   float64
 9   Measles                          2938 non-null   int64  
 10   BMI                             2904 non-null   float64
 11  under-five deaths                2938 non-null   int64  
 12  Polio                          

In [4]:
#список всех признаков по убыванию количества пропусков в них
data.isnull().sum().sort_values(ascending=False)

Population                         652
Hepatitis B                        553
GDP                                448
Total expenditure                  226
Alcohol                            194
Income composition of resources    167
Schooling                          163
 thinness  1-19 years               34
 thinness 5-9 years                 34
 BMI                                34
Diphtheria                          19
Polio                               19
Life expectancy                     10
Adult Mortality                     10
infant deaths                        0
Status                               0
Country                              0
Year                                 0
under-five deaths                    0
Measles                              0
percentage expenditure               0
 HIV/AIDS                            0
dtype: int64

In [5]:
#удаляем строки с пропуском в целевой переменной
data = data.dropna(subset = ['Life expectancy '])

In [6]:
y = data["Life expectancy "]
X = data.drop("Life expectancy ", axis = 1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

кодируем переменные

In [7]:
X["Status"].value_counts() #закодируем с помощью Label Encoding

Status
Developing    2416
Developed      512
Name: count, dtype: int64

In [8]:
X["Country"].nunique() #кодируем с FeatureHasher так как много различных значений

183

In [9]:
label_encoder = LabelEncoder()
X_train['Status'] = label_encoder.fit_transform(X_train['Status'])
X_test['Status'] = label_encoder.transform(X_test['Status'])

hash_encoder = FeatureHasher(n_features=8, input_type='string')

country_train = hash_encoder.fit_transform(X_train['Country'].apply(lambda x: [x])).toarray()
country_test = hash_encoder.transform(X_test['Country'].apply(lambda x: [x])).toarray()

df_country_train = pd.DataFrame(country_train, columns=[f'country_hash_{i}' for i in range(8)], index=X_train.index)
df_country_test = pd.DataFrame(country_test, columns=[f'country_hash_{i}' for i in range(8)], index=X_test.index)


X_train = X_train.drop(columns = 'Country').join(df_country_train)
X_test = X_test.drop(columns = 'Country').join(df_country_test)

In [10]:
simple_cols = ['Adult Mortality']

knn_cols = [' BMI ', 'Polio', 'Diphtheria ', ' thinness  1-19 years', ' thinness 5-9 years']

iter_cols = ['Alcohol', 'Total expenditure', 'Schooling', 'Income composition of resources', 'GDP', 'Hepatitis B', 'Population']


маштабируем данные (перед использованием  KNNImputer), используем imputer для заполнения пропусков в данных: (transformers)

In [11]:
scaler = StandardScaler() #масштабируем данные
knn_imputer = KNNImputer(n_neighbors=3)
simple_imputer = SimpleImputer(strategy='median')
iterative_imputer = IterativeImputer(max_iter=10, random_state=1)

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=X_test.columns,  index=X_test.index)

X_train_scaled[knn_cols] = knn_imputer.fit_transform(X_train_scaled[knn_cols])
X_test_scaled[knn_cols] = knn_imputer.transform(X_test_scaled[knn_cols])

X_train_scaled[simple_cols] = simple_imputer.fit_transform(X_train_scaled[simple_cols])
X_test_scaled[simple_cols] = simple_imputer.transform(X_test_scaled[simple_cols])

X_train_scaled[iter_cols] = iterative_imputer.fit_transform(X_train_scaled[iter_cols])
X_test_scaled[iter_cols] = iterative_imputer.transform(X_test_scaled[iter_cols])

In [12]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred = lr.predict(X_test_scaled)


Оценка модели:

In [13]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred)

print(f"MAE:  {mae:.2f}")
print(f"MSE:  {mse:.2f}")
print(f"MSE:  {r2:.2f}") #коэффициент детерминации
print(f"MAPE:  {mape:.2f}")

MAE:  2.89
MSE:  14.63
MSE:  0.83
MAPE:  0.04
